# NYMEX natural gas options from Databento
Run cells 1–3 (free apart from cell 2, a one-day definitions sample whose cost is printed and capped at $10), paste the cost table to Claude, then set PRODUCTS/BUDGET_USD in cell 4 and run 4–7. Everything is saved to Google Drive (`MyDrive/natgas_options`) and every cell is safe to rerun. Licensed data: keep it private (do not commit to GitHub).

In [ ]:
# %% [1] Setup: install, mount Drive, read your Databento key from Colab Secrets
!pip -q install databento pandas pyarrow
from google.colab import drive, userdata
drive.mount("/content/drive")
import re, time, json
import numpy as np, pandas as pd, databento as db
from pathlib import Path

client = db.Historical(userdata.get("DATABENTO_API_KEY"))
DS = "GLBX.MDP3"
OUT = Path("/content/drive/MyDrive/natgas_options"); OUT.mkdir(parents=True, exist_ok=True)
START = "2010-06-06"
END = (pd.Timestamp.now("UTC").normalize() - pd.Timedelta(days=3)).strftime("%Y-%m-%d")  # historical lag
print("Saving to", OUT, "| history", START, "->", END)

In [ ]:
# %% [2] Discover every NYMEX option product written on natural gas (NG) futures
# Reads ONE day of contract definitions for the whole CME Globex dataset, then keeps options whose
# underlying is an NG future or an NG calendar spread. The cost is printed first; it only proceeds if small.
DAY = ("2024-01-02", "2024-01-03")
c = client.metadata.get_cost(dataset=DS, schema="definition", symbols="ALL_SYMBOLS", start=DAY[0], end=DAY[1])
print(f"One day of all Globex definitions costs ${c:.2f}")
if c > 10:
    raise SystemExit("Too expensive for discovery - paste this output to Claude instead.")
defs = client.timeseries.get_range(dataset=DS, schema="definition", symbols="ALL_SYMBOLS",
                                   start=DAY[0], end=DAY[1]).to_df()
defs.to_parquet(OUT / "definitions_2024-01-02.parquet")
is_opt = defs["instrument_class"].astype(str).isin(["C", "P"])
on_ng = defs["underlying"].astype(str).str.contains(r"(^|[^A-Z])NG[FGHJKMNQUVXZ]\d", regex=True)
ng_opts = defs[is_opt & on_ng]
summary = (ng_opts.groupby("asset")
           .agg(n_instruments=("raw_symbol", "size"),
                example_symbol=("raw_symbol", "first"),
                example_underlying=("underlying", "first"),
                n_underlyings=("underlying", "nunique"))
           .sort_values("n_instruments", ascending=False))
print(summary.to_string())
# Wider net (e.g. calendar spread options whose underlying is written differently)
broad = defs[is_opt & ~on_ng & (defs["underlying"].astype(str).str.contains("NG")
                                | defs["raw_symbol"].astype(str).str.match(r"^(G\d|NG|LN|ON)"))]
if len(broad):
    print("\nOther options mentioning NG (check these):")
    print(broad.groupby("asset").agg(n=("raw_symbol", "size"), example_symbol=("raw_symbol", "first"),
                                     example_underlying=("underlying", "first")).to_string())
PRODUCTS_FOUND = [f"{a}.OPT" for a in summary.index] + [f"{a}.OPT" for a in broad["asset"].unique()]
print("\nOption products on NG:", PRODUCTS_FOUND)

In [ ]:
# %% [3] Price the full history of each product (free: nothing is downloaded or billed)
rows = []
for p in PRODUCTS_FOUND:
    for schema in ["statistics", "ohlcv-1d"]:
        try:
            usd = client.metadata.get_cost(dataset=DS, symbols=[p], stype_in="parent", schema=schema,
                                           start=START, end=END)
            gb = client.metadata.get_billable_size(dataset=DS, symbols=[p], stype_in="parent", schema=schema,
                                                   start=START, end=END) / 1e9
            rows.append((p, schema, round(usd, 2), round(gb, 2)))
        except Exception as e:
            rows.append((p, schema, None, str(e)[:60]))
costs = pd.DataFrame(rows, columns=["product", "schema", "usd", "GB"])
costs.to_csv(OUT / "cost_estimates.csv", index=False)
print(costs.to_string())
print("\nstatistics = daily settlement price, volume and open interest per option (what we need).")
print("Paste this table to Claude for a recommendation, or pick products in the next cell.")

In [ ]:
# %% [4] Choose what to buy and approve a budget. Nothing is billed unless total <= BUDGET_USD.
PRODUCTS = []          # e.g. ["LNE.OPT", "ON.OPT", "G4.OPT"] - fill from the table above
BUDGET_USD = 0         # the most you approve, in dollars
sel = costs[(costs["product"].isin(PRODUCTS)) & (costs["schema"] == "statistics")]
total = float(sel["usd"].sum())
print(sel.to_string(), f"\nTotal: ${total:.2f}  |  budget: ${BUDGET_USD}")
assert PRODUCTS, "Set PRODUCTS first"
assert total <= BUDGET_USD, "Total exceeds BUDGET_USD - raise the budget or drop products"

jobs_file = OUT / "jobs.json"
jobs = json.loads(jobs_file.read_text()) if jobs_file.exists() else {}
for p in PRODUCTS:
    if p in jobs:
        print(p, "already submitted:", jobs[p]); continue
    j = client.batch.submit_job(dataset=DS, symbols=[p], stype_in="parent", schema="statistics",
                                start=START, end=END, encoding="dbn", compression="zstd",
                                split_duration="month")
    jobs[p] = j["id"]; jobs_file.write_text(json.dumps(jobs, indent=1))
    print("submitted", p, j["id"])

In [ ]:
# %% [5] Wait for the jobs and download to Drive (safe to rerun; finished downloads are kept)
jobs = json.loads((OUT / "jobs.json").read_text())
pending = dict(jobs)
while pending:
    try:
        states = {j["id"]: j["state"] for j in client.batch.list_jobs(since="2026-11-01")}
    except Exception as e:
        print("status check failed, retrying in 60s:", type(e).__name__); time.sleep(60); continue
    for p, jid in list(pending.items()):
        st = states.get(jid, "unknown")
        if st == "done":
            dest = OUT / "raw" / p.replace(".", "_")
            if not list(dest.rglob("*.dbn.zst")):
                for attempt in range(5):
                    try:
                        client.batch.download(job_id=jid, output_dir=dest); break
                    except Exception as e:
                        print("download retry", p, type(e).__name__); time.sleep(60)
            print(p, "downloaded:", len(list(dest.rglob("*.dbn.zst"))), "files")
            pending.pop(p)
        elif st in ("expired", "unknown"):
            print(p, "job", jid, "is", st); pending.pop(p)
        else:
            print(p, st)
    if pending:
        time.sleep(90)
print("All downloads finished.")

In [ ]:
# %% [6] Compact: daily settle / volume / open interest per option -> Parquet per product (on Drive)
KEEP = {db.StatType.SETTLEMENT_PRICE: "settle", db.StatType.CLEARED_VOLUME: "volume",
        db.StatType.OPEN_INTEREST: "oi"}
for p in json.loads((OUT / "jobs.json").read_text()):
    src = OUT / "raw" / p.replace(".", "_")
    dst = OUT / f"compact_{p.replace('.', '_')}.parquet"
    if dst.exists():
        print("exists", dst.name); continue
    parts = []
    for f in sorted(src.rglob("*.dbn.zst")):
        for df in db.DBNStore.from_file(str(f)).to_df(count=2_000_000, map_symbols=True):
            df = df[df["stat_type"].isin(list(KEEP))]
            if df.empty:
                continue
            parts.append(pd.DataFrame({
                "ts_recv": df.index,
                # ts_ref is 00:00 UTC on the trading date: take the UTC date (no time-zone shift)
                "date": pd.to_datetime(df["ts_ref"]).dt.tz_convert("UTC").dt.normalize().dt.tz_localize(None).values,
                "symbol": df["symbol"].astype(str).values,
                "field": df["stat_type"].map(KEEP).values,
                "value": np.where(df["stat_type"] == db.StatType.SETTLEMENT_PRICE, df["price"], df["quantity"]),
            }))
        print(p, f.name, "rows so far:", sum(len(x) for x in parts))
    long = pd.concat(parts).sort_values("ts_recv")
    last = long.groupby(["date", "symbol", "field"], as_index=False)["value"].last()  # final revision
    wide = last.pivot_table(index=["date", "symbol"], columns="field", values="value").reset_index()
    wide.to_parquet(dst, index=False)
    print("saved", dst.name, len(wide), "option-days")

In [ ]:
# %% [7] Reduce to a small file for Claude: strikes near the money, expiries within 18 months
# The underlying price is inferred from put-call parity (F = K + C - P at the strike where C - P is
# smallest), so no futures data is needed here. Symbols look like 'LNEH4 C2500'.
PAT = re.compile(r"^(?P<root>[A-Z0-9]+?)(?P<m>[FGHJKMNQUVXZ])(?P<y>\d{1,2})\s+(?P<cp>[CP])(?P<k>-?\d+)$")
def parse(sym):
    m = PAT.match(sym.strip())
    return (m["root"], m["m"], int(m["y"]), m["cp"], int(m["k"])) if m else (None,) * 5
out = []
for f in sorted(OUT.glob("compact_*.parquet")):
    w = pd.read_parquet(f)
    w = w[w["settle"].notna() & (w["settle"] > 0)]
    p = pd.DataFrame([parse(s) for s in w["symbol"]], columns=["root", "mcode", "y", "cp", "k_raw"], index=w.index)
    w = w.join(p).dropna(subset=["root"])
    yr = w["date"].dt.year
    y = w["y"].astype(int)
    full = np.where(y >= 10, 2000 + y, (yr // 10) * 10 + y)          # 1-digit years: same decade...
    full = np.where(full < yr - 1, full + 10, full)                     # ...or the next one
    w["cy"] = full; w["cm"] = w["mcode"].map({c: i + 1 for i, c in enumerate("FGHJKMNQUVXZ")})
    w = w[(w["cy"] * 12 + w["cm"]) - (yr * 12 + w["date"].dt.month) <= 18]
    piv = w.pivot_table(index=["date", "root", "cy", "cm", "k_raw"], columns="cp", values="settle")
    piv = piv.dropna().reset_index()
    piv["cp_diff"] = (piv["C"] - piv["P"]).abs()
    atm = piv.loc[piv.groupby(["date", "root", "cy", "cm"])["cp_diff"].idxmin()]
    atm = atm.rename(columns={"k_raw": "k_atm_raw"})
    w = w.merge(atm[["date", "root", "cy", "cm", "k_atm_raw"]], on=["date", "root", "cy", "cm"])
    band = (w["k_raw"] - w["k_atm_raw"]).abs() <= 0.4 * w["k_atm_raw"].abs().clip(lower=100)
    keep = w[band][["date", "root", "cy", "cm", "cp", "k_raw", "settle", "volume", "oi"]]
    out.append(keep); print(f.name, "->", len(keep), "rows near the money")
small = pd.concat(out).sort_values(["date", "root", "cy", "cm", "cp", "k_raw"])
path = OUT / "ng_options_near_money.csv.gz"
small.to_csv(path, index=False, compression="gzip")
print("saved", path, f"{path.stat().st_size/1e6:.1f} MB,", len(small), "rows,",
      small["date"].min().date(), "->", small["date"].max().date())
print(small.groupby("root").size())